# Metro Traffic Data Preprocessing

This notebook preprocesses the metro traffic dataset for the CityFlow ML project, preparing it for Tension Score prediction.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## Load Data

In [2]:
df = pd.read_csv('../data/metro_traffic.csv')

print("Original dataset shape:", df.shape)
print("Original columns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Original dataset shape: (48204, 9)
Original columns: ['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description', 'date_time', 'traffic_volume']

First 5 rows:


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


## Data Cleaning and Feature Extraction

In [3]:
df_clean = df.copy()

# Convert date_time to datetime
df_clean['date_time'] = pd.to_datetime(df_clean['date_time'])

# Extract temporal features
df_clean['hour'] = df_clean['date_time'].dt.hour
df_clean['day_of_week'] = df_clean['date_time'].dt.dayofweek  # 0=Monday, 6=Sunday
df_clean['is_weekend'] = df_clean['day_of_week'].isin([5, 6]).astype(int)  # 5=Saturday, 6=Sunday

# Drop the mostly empty holiday column (99.8% missing)
df_clean = df_clean.drop('holiday', axis=1)

# Convert temperature from Kelvin to Celsius
df_clean['temp_celsius'] = df_clean['temp'] - 273.15

print("After feature extraction:")
print("Shape:", df_clean.shape)
print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

After feature extraction:
Shape: (48204, 12)

Missing values after cleaning:
temp                   0
rain_1h                0
snow_1h                0
clouds_all             0
weather_main           0
weather_description    0
date_time              0
traffic_volume         0
hour                   0
day_of_week            0
is_weekend             0
temp_celsius           0
dtype: int64


## Feature Engineering and Target Creation

In [4]:
# Create cyclical features for hour (time is cyclical - 23h is close to 0h)
df_clean['hour_sin'] = np.sin(2 * np.pi * df_clean['hour'] / 24)
df_clean['hour_cos'] = np.cos(2 * np.pi * df_clean['hour'] / 24)

# Handle outliers in temperature using IQR method
print("Handling outliers...")
Q1 = df_clean['temp_celsius'].quantile(0.25)
Q3 = df_clean['temp_celsius'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR
outliers_count = ((df_clean['temp_celsius'] < lower_bound) | (df_clean['temp_celsius'] > upper_bound)).sum()
df_clean['temp_celsius'] = df_clean['temp_celsius'].clip(lower=lower_bound, upper=upper_bound)
print(f"temp_celsius: Capped {outliers_count} outliers")

# Cap rain and snow at physically realistic maximum (50mm/h)
# Preserves zeros and moderate values, only caps extreme outliers/errors
df_clean['rain_1h'] = df_clean['rain_1h'].clip(upper=50)
df_clean['snow_1h'] = df_clean['snow_1h'].clip(upper=50)
print("rain_1h & snow_1h: Capped values > 50mm/h (extreme weather preserved)")

# Normalize continuous features (mean=0, std=1)
continuous_features = ['temp_celsius', 'rain_1h', 'snow_1h']
scaler_features = StandardScaler()
df_clean[continuous_features] = scaler_features.fit_transform(df_clean[continuous_features])

# Create target: Normalize traffic_volume to tension_score (0-100)
scaler_target = MinMaxScaler(feature_range=(0, 100))
df_clean['tension_score'] = scaler_target.fit_transform(df_clean[['traffic_volume']])

print(f"\nTension score range: [{df_clean['tension_score'].min():.1f}, {df_clean['tension_score'].max():.1f}]")

# Select final features for model
feature_cols = ['hour_sin', 'hour_cos', 'day_of_week', 'is_weekend', 'temp_celsius', 'rain_1h', 'snow_1h']
target_col = 'tension_score'

# Create final dataset
df_final = df_clean[feature_cols + [target_col]]

print(f"\nFinal dataset: {df_final.shape}")
print(f"Features: {feature_cols}")
print(f"Target: {target_col}")

Handling outliers...
temp_celsius: Capped 10 outliers
rain_1h & snow_1h: Capped values > 50mm/h (extreme weather preserved)

Tension score range: [0.0, 100.0]

Final dataset: (48204, 8)
Features: ['hour_sin', 'hour_cos', 'day_of_week', 'is_weekend', 'temp_celsius', 'rain_1h', 'snow_1h']
Target: tension_score


## Save Cleaned Dataset

In [5]:
# Save the cleaned dataset
output_path = '../data/metro_traffic_cleaned.csv'
df_final.to_csv(output_path, index=False)

print(f"\nCleaned dataset saved to: {output_path}")
print(f"Shape: {df_final.shape}")

# Save preprocessing objects for use in training/prediction
import joblib
preprocessing_info = {
    'feature_cols': feature_cols,
    'target_col': target_col,
    'scaler_target': scaler_target,
    'scaler_features': scaler_features
}

joblib.dump(preprocessing_info, '../models/preprocessing_info.joblib')
print("Preprocessing info saved to: ../models/preprocessing_info.joblib")

print("\nPreprocessing completed successfully!")


Cleaned dataset saved to: ../data/metro_traffic_cleaned.csv
Shape: (48204, 8)
Preprocessing info saved to: ../models/preprocessing_info.joblib

Preprocessing completed successfully!
